# admdongkor — 사용 예시

한국 행정경계 (읍면동/시군구/시도) **1975–2026 시계열** 다루기.

<a target="_blank" href="https://colab.research.google.com/github/vuski/admdongkor/blob/master/examples/example.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

이 노트북은 Colab 에서 위 버튼으로 바로 열거나 로컬 Jupyter 에서 돌릴 수 있다.

**목차**
1. 환경 준비 (설치 + 한글 폰트)
2. 버전 탐색 — `versions()`
3. 이름으로 찾기 — `find()`
4. 지도 받기 — `get()` (EPSG:5179 기본 / WGS84 옵션)
5. 영역 시계열 매칭 — `match_adm()`
6. 두 시점 diff — `compare()`
7. 실전 예: 인구 집계를 위한 코드 매핑


## 1. 환경 준비


In [ ]:
# Colab 에서 처음 실행할 때만: 라이브러리 + 한글 폰트 설치
import sys
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    !pip install -q admdongkor
    !apt-get -qq install -y fonts-nanum
    !fc-cache -fv > /dev/null


In [ ]:
import glob
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import font_manager

# 한글 폰트
if sys.platform == 'win32':
    matplotlib.rcParams['font.family'] = 'Malgun Gothic'
elif sys.platform == 'darwin':
    matplotlib.rcParams['font.family'] = 'AppleGothic'
else:
    # Linux (Colab 포함). apt 로 설치된 폰트 직접 주입 (matplotlib 캐시 우회).
    for path in glob.glob('/usr/share/fonts/**/Nanum*.ttf', recursive=True) + \
                glob.glob('/usr/share/fonts/**/NotoSansCJK*.ttc', recursive=True):
        font_manager.fontManager.addfont(path)
    for cand in ('NanumGothic', 'Noto Sans CJK KR', 'UnDotum'):
        if any(cand in f.name for f in font_manager.fontManager.ttflist):
            matplotlib.rcParams['font.family'] = cand
            break
matplotlib.rcParams['axes.unicode_minus'] = False
print('font:', matplotlib.rcParams['font.family'])

import admdongkor as adk
print('admdongkor version:', adk.__version__)


## 2. 버전 탐색 — `versions()`

1975–2026 총 61 개 버전. `YYYY1231` (shapefile 기반, 1975-2015) 과
`YYYYMMDD` (GeoJSON 기반, 2012-2026) 두 포맷이 섞여 있다.


In [ ]:
print('total:', len(adk.versions()))
print('head :', adk.versions().head())
print('tail :', adk.versions().tail())
print('2023 :', adk.versions(2023))


## 3. 이름으로 버전 찾기 — `find()`


In [ ]:
# 단일 토큰: 전 레벨에서 '종로' 부분일치
adk.find('종로').head(8)


In [ ]:
# 2 토큰: 자동으로 sgg 만 (읍면동 줄줄이 안 나옴)
adk.find('서울특별시 종로구').head()


In [ ]:
# 공백 무시 — '수원시 권선구' 가 '수원시권선구' 와도 매치
adk.find('수원시 권선구').head()


In [ ]:
# 체이닝 — 여주군은 1975~2012 경기도 시절, 2013 여주시로 승격
r = adk.find('여주군')
print('versions:', r.versions())
print('first   :', r.first())
print('last    :', r.last(), '(2013 여주시로 승격 직전)')


## 4. 지도 받기 — `get()`


In [ ]:
# 기본: EPSG:5179 (Korea 2000 / Unified CS) — 면적·거리 계산 정확
sido = adk.get('20250401', 'sido')
print('CRS :', sido.crs)
print('rows:', len(sido))

fig, ax = plt.subplots(figsize=(7, 9))
sido.plot(ax=ax, edgecolor='black', linewidth=0.4, facecolor='lightgrey')
ax.set_title('시도 — 20250401 (EPSG:5179)')
ax.set_axis_off()


In [ ]:
# crs= 옵션으로 WGS84 재투영. Folium/Leaflet 등 웹지도용.
sido_wgs = adk.get('20250401', 'sido', crs='EPSG:4326')
print('CRS   :', sido_wgs.crs)
print('bounds:', sido_wgs.total_bounds, '(경도 124-132, 위도 33-43 범위)')


## 5. 영역 시계열 매칭 — `match_adm()`

**핵심 기능**. 어느 시점(`base`)의 특정 영역(`region`) 을 기준으로,
다른 시점(`target`) 에서 그 영역에 걸치는 읍면동들을 가중치와 함께 반환.

**사례: 2023년 군위군 대구 편입.** 2025 대구(sidocd=27) 영역을 2011 시점에 매칭하면,
당시 대구 전체 + **경북(47) 군위군** 이 같이 나와야 한다.


In [ ]:
r = adk.match_adm(base='20251231', region='27', target='20111231')
print(f'rows: {len(r)}')
r.head()


In [ ]:
# 경북(sidocd=47) 군위군(sggcd=47720) 읍면들이 포함됐는지
gunwi = r[r.sggcd == '47720']
print(f'군위군 읍면: {len(gunwi)}개')
gunwi[['emdcd', 'emdnm', 'sggnm', 'sidonm', 'weight']]


In [ ]:
# 레벨별 집계 — `.sgg()` / `.sido()` 는 면적가중 평균
print('=== sido() ===')
print(r.sido())
print()
print('=== sgg() — 대구 안 전체 sgg + 군위 ===')
print(r.sgg().head(12))


In [ ]:
# 시각화: 2011 지도 위에 matched emd 를 weight 로 칠하고 sgg/sido 테두리
emd_2011 = adk.get('20111231', 'emd')
sgg_2011 = adk.get('20111231', 'sgg')
sido_2011 = adk.get('20111231', 'sido')

matched = emd_2011[emd_2011.emdcd.isin(r.emdcd)].merge(r[['emdcd','weight']], on='emdcd')
matched_sgg = sgg_2011[sgg_2011.sggcd.isin(matched.sggcd.unique())]
matched_sido = sido_2011[sido_2011.sidocd.isin(matched.sidocd.unique())]

fig, ax = plt.subplots(figsize=(9, 10))
matched.plot(ax=ax, column='weight', cmap='Reds', vmin=0, vmax=1,
             edgecolor='none', legend=True, alpha=0.9)
matched_sgg.plot(ax=ax, facecolor='none', edgecolor='black', linewidth=0.8)
matched_sido.plot(ax=ax, facecolor='none', edgecolor='black', linewidth=1.8)
ax.set_title('2025 대구광역시 영역 → 2011 기준 매칭\n(경북 군위군이 포함돼 있음)')
ax.set_axis_off()


In [ ]:
# 여러 target 을 한 번에 — 연도별 sido 집계 비교
r_multi = adk.match_adm(
    base='20251231', region='27',
    target=['20111231', '20201001', '20241231'],
)
r_multi.sido()


## 6. 두 시점 비교 — `compare()`


In [ ]:
# 2011 vs 2013 — 세종시 신설 (2012-07) 이 경계에 어떻게 잡히는지
c = adk.compare(['20111231', '20131231'])
print(c)

# 세종 emd 가 only_in_b (= 2013 에 신설) 로 나와야
sejong = c.diff()[c.diff().sidonm.astype(str).str.contains('세종', na=False)]
print(f'\n세종 emd (only_in_b 예상): {len(sejong)}개')
sejong.head()


In [ ]:
# 경계가 가장 많이 변한 emd top 10
c25_11 = adk.compare(['20251231', '20111231'])
changed = c25_11.diff()[c25_11.diff().status == 'changed'].drop_duplicates('emdcd')
changed.nsmallest(10, 'iou')[['emdcd', 'emdnm', 'sggnm', 'sidonm', 'iou']]


## 7. 실전 예: 인구 데이터 시계열 집계

시계열 인구 데이터를 **고정된 행정구역 영역** (예: 현재 대구)으로 집계할 때,
각 시점에 그 영역에 속했던 읍면동 + 비율(weight) 을 알아야 한다. 그게 바로 `match_adm()`.

**사용 패턴:**
```python
# 가상 2011 emd별 인구 테이블 (emdcd, population)
pop_2011 = load_my_population_data('2011')

# 2025 대구 영역에 대응하는 2011 emd + weight
mapping = adk.match_adm(base='20251231', region='27', target='20111231')

# 가중 합: 대구(2025 영역) 에 해당하는 2011 인구 추정
daegu_2011_pop = (mapping
    .merge(pop_2011, on='emdcd')
    .assign(contrib=lambda d: d.population * d.weight)
    .contrib.sum())
```

아래는 위 공식을 실측 시뮬레이션으로 확인하는 셀 (가짜 인구: 면적 비례로 생성).


In [ ]:
# 가상 인구 생성: 2011 emd 별 '면적 × 1/㎡당 10명' 가정
import pandas as pd
emd_2011_info = emd_2011[['emdcd','area']].copy()
emd_2011_info['population'] = (emd_2011_info['area'] / 1e4 * 10).round().astype(int)

# 매핑 + 가중합
mapping = adk.match_adm(base='20251231', region='27', target='20111231')
joined = mapping.merge(emd_2011_info[['emdcd','population']], on='emdcd')
joined['contrib'] = joined['population'] * joined['weight']
print(f'2011 시점 기준, 2025 대구 영역의 추정 인구: {joined.contrib.sum():,.0f} 명')
print(f'비교 (가중 없이 단순 합): {joined.population.sum():,.0f} 명 — 과대집계')

# 경북 소속(군위)만 골라보면
only_gb = joined[joined.sidocd == '47']
print(f'\n그 중 경북(2011 군위) 기여: {only_gb.contrib.sum():,.0f} 명')


---

더 깊은 내용은 [GitHub 레포](https://github.com/vuski/admdongkor) 와 `lib/README.md`.
